In [ ]:
﻿# %% [markdown]
# # Notebook 4: Monitoring & Learning Agent
# **EV Dynamic Tariff Optimization -- Agentic AI Framework**
#
# This notebook implements **Agent 3: the Monitoring & Learning Agent**.
#
# The agent's role is to:
# 1. Evaluate outcomes of pricing decisions made by Agent 2
# 2. Track key performance metrics across multiple simulation episodes
# 3. Adjust pricing thresholds through a feedback loop to improve over time
#
# This simulates a real-world deployment where the system continuously learns.


## 1. Setup


In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (12, 6), 'font.size': 11, 'axes.grid': True,
    'grid.alpha': 0.3, 'figure.dpi': 100
})

COLORS = {'peak': '#e74c3c', 'shoulder': '#f39c12', 'off_peak': '#2ecc71',
          'primary': '#3498db', 'secondary': '#9b59b6', 'dark': '#2c3e50'}

def get_project_root():
    """Find the project root whether this runs as a .py file or notebook."""
    start_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()
    current = os.path.abspath(start_dir)
    for _ in range(5):
        if os.path.isdir(os.path.join(current, 'data')) and os.path.isdir(os.path.join(current, 'notebooks')):
            return current
        current = os.path.dirname(current)
    return os.path.abspath(os.path.join(os.getcwd(), '..'))

BASE_DIR      = get_project_root()
PROCESSED_DIR = os.path.join(BASE_DIR, 'data', 'processed')
OUTPUT_DIR    = os.path.join(BASE_DIR, 'outputs')


## 2. Load Data and Previous Agent Outputs


In [ ]:
acn        = pd.read_csv(os.path.join(PROCESSED_DIR, 'acn_sessions_processed.csv'),
                          parse_dates=['connection_dt', 'disconnect_dt'])
acn_hourly = pd.read_csv(os.path.join(PROCESSED_DIR, 'acn_hourly_demand.csv'),
                          parse_dates=['datetime'])

# Load UrbanEV city-level hourly data (produced by preprocessing notebook)
urban_city_hourly_path = os.path.join(PROCESSED_DIR, 'urbanev_city_hourly.csv')
urban_hourly_zone_path = os.path.join(PROCESSED_DIR, 'urbanev_hourly.csv')

if os.path.exists(urban_city_hourly_path):
    urban_hourly = pd.read_csv(urban_city_hourly_path, parse_dates=['datetime'])
    print("Loaded urbanev_city_hourly.csv")
else:
    urban_hourly_zone = pd.read_csv(urban_hourly_zone_path, parse_dates=['datetime'])
    urban_hourly = urban_hourly_zone.groupby('datetime').agg(
        active_sessions=('occupancy', 'sum'),
        avg_utilization=('utilization_rate', 'mean'),
        queue_proxy=('queue_proxy', 'sum'),
        hour=('hour', 'first'),
        day_of_week=('day_of_week', 'first'),
    ).reset_index()
    print("Computed city-level hourly from zone data (fallback)")

acn_tariffs = pd.read_csv(os.path.join(OUTPUT_DIR, 'dynamic_tariffs_acn.csv'),
                           parse_dates=['connection_dt'])

urban_tariffs_path = os.path.join(OUTPUT_DIR, 'dynamic_tariffs_urbanev.csv')
if os.path.exists(urban_tariffs_path):
    urban_tariffs = pd.read_csv(urban_tariffs_path, parse_dates=['datetime'])
else:
    urban_tariffs = None

# Load Agent 1+2 KPI output to get the actual off-peak uplift number
agent12_kpi_path = os.path.join(OUTPUT_DIR, 'agent1_agent2_kpi.csv')
if os.path.exists(agent12_kpi_path):
    agent12_kpi = pd.read_csv(agent12_kpi_path)
    off_peak_row = agent12_kpi[agent12_kpi['KPI'].str.contains('Off-Peak', case=False)]
    OFF_PEAK_UPLIFT_FROM_NB3 = off_peak_row['Value'].values[0] if len(off_peak_row) > 0 else "N/A"
else:
    OFF_PEAK_UPLIFT_FROM_NB3 = "N/A"

print(f"ACN sessions:    {len(acn):,}")
print(f"ACN hourly:      {len(acn_hourly):,}")
print(f"ACN tariffs:     {len(acn_tariffs):,}")
print(f"UrbanEV hourly:  {len(urban_hourly):,}")
print(f"UrbanEV tariffs: {len(urban_tariffs):,}" if urban_tariffs is not None else "UrbanEV tariffs: not found")
print(f"Off-Peak Uplift (from Notebook 3): {OFF_PEAK_UPLIFT_FROM_NB3}")


---
## 3. Define Monitoring Metrics

KPIs tracked per episode:
1. **Posted Revenue Gain %** -- dynamic vs flat baseline before demand response (ACN, USD)
2. **Net Revenue Gain %** -- dynamic vs flat baseline after demand response (ACN, USD)
3. **Customer Response Rate** -- demand retained after elasticity-based response
4. **Pricing Efficiency Score** -- net revenue per adjusted kWh delivered (ACN, USD/kWh)
5. **Average Waiting Time Reduction** -- UrbanEV congested-hour utilization reduction (queue proxy)


### 3.1 Pricing Function (parameterized for feedback loop)


In [ ]:
def dynamic_pricing_agent(utilization, base_rate, surge_threshold, discount_threshold,
                           max_surge=1.40, max_discount=0.85, time_period='shoulder'):
    """
    ACN-calibrated pricing agent with adjustable thresholds for learning.
    max_discount=0.85 (only -15% max) because ACN is low-utilization by nature.
    surge_threshold starts at 0.65 consistent with Notebook 3.
    """
    if utilization > surge_threshold:
        surge_intensity = min((utilization - surge_threshold) / (1.0 - surge_threshold), 1.0)
        multiplier = 1.0 + (max_surge - 1.0) * surge_intensity
        tier = 'surge'
    elif utilization < discount_threshold:
        discount_intensity = min((discount_threshold - utilization) / discount_threshold, 1.0)
        multiplier = 1.0 - (1.0 - max_discount) * discount_intensity
        tier = 'discount'
    else:
        if time_period == 'peak':
            multiplier = 1.10
        elif time_period == 'off_peak':
            multiplier = 0.97
        else:
            multiplier = 1.00
        tier = 'normal'

    return base_rate * multiplier, multiplier, tier


### 3.2 Demand Elasticity Model

**Assumption:** Price elasticity of demand = **-0.30**
A 10% price increase -> about 3% demand decrease.
Source: Consistent with empirical EV charging studies (conservative estimate).


In [ ]:
PRICE_ELASTICITY = -0.30

def simulate_demand_response(original_demand, price_multiplier, elasticity=PRICE_ELASTICITY):
    """
    Simulate how energy demand changes in response to a price change.
    demand_change_pct = elasticity * (price_multiplier - 1)
    """
    price_change  = price_multiplier - 1.0
    demand_change = elasticity * price_change
    return max(0, original_demand * (1 + demand_change))

print("=== Demand Elasticity Demo ===")
for mult in [0.70, 0.85, 1.00, 1.15, 1.30, 1.40]:
    new_d = simulate_demand_response(100, mult)
    print(f"  Price {mult:.2f}x -> Demand: {new_d:.1f} (change: {new_d-100:+.1f}%)")


---
## 4. Build UrbanEV Congestion Profile for Wait Time Metric

**Correct wait-time proxy:**

The project does not contain observed queue/wait-time labels, so we use UrbanEV
utilization as a congestion proxy: higher utilization means higher queue risk.

The previous version used fixed "peak hours" that did not match the UrbanEV data.
Here, congested hours are learned from UrbanEV itself: the top quartile of hourly
baseline utilization. We then compare baseline utilization with the simulated
utilization produced by Agent 2's UrbanEV pricing output.


In [ ]:
if urban_tariffs is not None and {'utilization_rate', 'predicted_utilization', 'time_period'}.issubset(urban_tariffs.columns):
    urban_tariffs['hour'] = pd.to_datetime(urban_tariffs['datetime']).dt.hour
    urban_wait_by_hour = urban_tariffs.groupby('hour').agg(
        baseline_utilization=('utilization_rate', 'mean')
    )
    congestion_cutoff = urban_wait_by_hour['baseline_utilization'].quantile(0.75)
    URBAN_CONGESTED_HOURS = urban_wait_by_hour[
        urban_wait_by_hour['baseline_utilization'] >= congestion_cutoff
    ].index.tolist()

    baseline_peak_congestion = urban_wait_by_hour.loc[
        URBAN_CONGESTED_HOURS, 'baseline_utilization'
    ].mean()

    print("Using UrbanEV tariff table as the wait-time proxy base")
    print("\nUrbanEV hourly congestion profile (baseline utilization):")
    print(urban_wait_by_hour.round(4).to_string())
    print(f"\nUrbanEV congested hours: {URBAN_CONGESTED_HOURS}")
    print(f"Congested-hour cutoff: {congestion_cutoff:.4f} avg utilization")
else:
    if 'avg_utilization' in urban_hourly.columns:
        urban_congestion_by_hour = urban_hourly.groupby('hour')['avg_utilization'].mean()
    else:
        max_sessions = urban_hourly['active_sessions'].max()
        urban_hourly['avg_utilization'] = (urban_hourly['active_sessions'] / max_sessions).clip(0, 1)
        urban_congestion_by_hour = urban_hourly.groupby('hour')['avg_utilization'].mean()

    congestion_cutoff = urban_congestion_by_hour.quantile(0.75)
    URBAN_CONGESTED_HOURS = urban_congestion_by_hour[
        urban_congestion_by_hour >= congestion_cutoff
    ].index.tolist()
    baseline_peak_congestion = urban_congestion_by_hour.loc[URBAN_CONGESTED_HOURS].mean()

    print("UrbanEV tariff output missing; using baseline congestion only")
    print("\nUrbanEV hourly congestion profile (avg utilization by hour):")
    print(urban_congestion_by_hour.round(4).to_string())

def evaluate_urbanev_wait_proxy(surge_thr, discount_thr):
    """
    Recompute UrbanEV congestion response for one monitoring policy.
    The official UrbanEV surge trigger remains at 80%; the learned surge
    threshold controls how strongly already-congested hours are discouraged.
    """
    if urban_tariffs is None or not {'utilization_rate', 'predicted_utilization', 'time_period'}.issubset(urban_tariffs.columns):
        return baseline_peak_congestion, baseline_peak_congestion, 0.0

    ep_urban = urban_tariffs.copy()
    util = ep_urban['predicted_utilization'].to_numpy()
    base_util = ep_urban['utilization_rate'].to_numpy()
    time_period = ep_urban['time_period'].to_numpy()
    congested_mask = ep_urban['hour'].isin(URBAN_CONGESTED_HOURS).to_numpy()

    multiplier = np.ones(len(ep_urban))
    surge_mask = util > 0.80
    discount_mask = (util < discount_thr) & (~congested_mask)

    if surge_mask.sum() > 0:
        surge_intensity = np.minimum((util[surge_mask] - 0.80) / 0.20, 1.0)
        multiplier[surge_mask] = 1.0 + (1.40 - 1.0) * surge_intensity

    congestion_guard_mask = congested_mask & (~surge_mask)
    congestion_guard_multiplier = min(1.12, 1.02 + (0.75 - surge_thr) * 0.25)
    multiplier[congestion_guard_mask] = congestion_guard_multiplier

    if discount_mask.sum() > 0:
        discount_intensity = np.minimum((discount_thr - util[discount_mask]) / discount_thr, 1.0)
        multiplier[discount_mask] = 1.0 - (1.0 - 0.70) * discount_intensity

    normal_mask = ~(surge_mask | discount_mask | congestion_guard_mask)
    multiplier[normal_mask & (time_period == 'peak')] = 1.03
    multiplier[normal_mask & (time_period == 'off_peak')] = 1.00

    ep_urban['simulated_utilization_episode'] = (
        base_util * (1 + PRICE_ELASTICITY * (multiplier - 1.0))
    ).clip(0, 1)

    wait_by_hour = ep_urban.groupby('hour').agg(
        baseline=('utilization_rate', 'mean'),
        post_pricing=('simulated_utilization_episode', 'mean')
    )
    congested_wait = wait_by_hour.loc[URBAN_CONGESTED_HOURS]
    baseline_wait = congested_wait['baseline'].mean()
    post_pricing_wait = congested_wait['post_pricing'].mean()
    wait_reduction = (
        (baseline_wait - post_pricing_wait) / baseline_wait * 100
    ) if baseline_wait > 0 else 0.0

    return baseline_wait, post_pricing_wait, wait_reduction

initial_baseline_wait, initial_post_wait, initial_wait_reduction = evaluate_urbanev_wait_proxy(0.70, 0.20)
print(f"\nBaseline congested-hour wait index:       {initial_baseline_wait:.4f}")
print(f"Initial post-pricing wait index:          {initial_post_wait:.4f}")
print(f"Initial waiting time reduction proxy:     {initial_wait_reduction:+.2f}%")


---
## 5. Multi-Episode Simulation with Feedback Loop

The agent runs **10 episodes**. After each episode it evaluates the metrics and
adjusts surge/discount thresholds using a bounded local search across 9 candidate
policies. The objective function rewards net revenue and congestion reduction while
penalizing excessive customer disruption.

**How Wait Time Reduction works here:**
- Dataset: UrbanEV only, because this KPI is defined as an UrbanEV queue proxy
- Congested hours: top quartile of UrbanEV hourly baseline utilization
- Wait reduction proxy = reduction in congested-hour utilization after Agent 2 pricing
- If the value is negative, the pricing policy increased congestion and must be fixed

**Customer Response Rate** = demand retained after elasticity.
**Price Exposure Rate** = % of sessions where price deviated from flat baseline.
The feedback loop should improve revenue and congestion while keeping exposure bounded.


In [ ]:
# --- Prepare ACN data ---
# Compute utilization proxy per session (based on hour-of-day average)
max_capacity = acn_hourly['active_sessions'].max()  # use true max, not 95th percentile
print(f"ACN true max capacity: {max_capacity:.0f} simultaneous sessions")

acn_hourly['utilization_proxy'] = (acn_hourly['active_sessions'] / max_capacity).clip(0, 1)
acn_util_map = acn_hourly.groupby('hour')['utilization_proxy'].mean().to_dict()
acn['estimated_utilization'] = acn['hour'].map(acn_util_map).fillna(0.3)

print(f"\nACN estimated utilization by hour:")
for h, v in sorted(acn_util_map.items()):
    print(f"  Hour {h:2d}: {v:.4f}")

BASELINE_RATE = 0.30  # USD/kWh (ACN)
# BASELINE NOTE: Problem statement specifies Rs.15/kWh for Indian market context.
# ACN is a US workplace dataset so we use the equivalent US rate of $0.30/kWh.
# Revenue Gain % methodology is identical regardless of currency.

N_EPISODES = 10

# Starting thresholds are calibrated for the ACN hourly-utilization distribution.
surge_threshold    = 0.70
discount_threshold = 0.20

PEAK_HOURS_ACN = [7, 8, 9, 10, 17, 18, 19, 20]  # morning + evening rush (ACN / US context)

print(f"\nStarting surge threshold:    {surge_threshold}")
print(f"Starting discount threshold: {discount_threshold}")
print(f"Episodes to simulate:        {N_EPISODES}")
print(f"Baseline wait index:         {initial_baseline_wait:.4f}")
print(f"Initial post-pricing index:  {initial_post_wait:.4f}")


### 5.1 Run the Feedback Loop


In [ ]:
episode_results = []

def evaluate_thresholds(surge_thr, discount_thr):
    """Evaluate one threshold policy and return all monitoring metrics."""
    ep_data = acn.copy()

    utilization       = ep_data['estimated_utilization'].to_numpy()
    time_period       = ep_data['time_period'].to_numpy()

    surge_mask        = utilization > surge_thr
    discount_mask     = utilization < discount_thr
    peak_period_mask  = (time_period == 'peak')
    offpeak_period_mask = (time_period == 'off_peak')

    multiplier = np.ones(len(ep_data))
    tier       = np.full(len(ep_data), 'normal', dtype=object)

    # Surge pricing: proportional between threshold and max
    surge_intensity    = np.zeros(len(ep_data))
    discount_intensity = np.zeros(len(ep_data))

    if surge_mask.sum() > 0:
        surge_intensity[surge_mask] = np.minimum(
            (utilization[surge_mask] - surge_thr) / (1.0 - surge_thr), 1.0
        )
        multiplier[surge_mask] = 1.0 + (1.40 - 1.0) * surge_intensity[surge_mask]
        tier[surge_mask] = 'surge'

    if discount_mask.sum() > 0:
        discount_intensity[discount_mask] = np.minimum(
            (discount_thr - utilization[discount_mask]) / discount_thr, 1.0
        )
        multiplier[discount_mask] = 1.0 - (1.0 - 0.85) * discount_intensity[discount_mask]
        tier[discount_mask] = 'discount'

    # Normal-zone time-of-day adjustment
    normal_mask = ~(surge_mask | discount_mask)
    multiplier[normal_mask & peak_period_mask]    = 1.10
    multiplier[normal_mask & offpeak_period_mask] = 0.97

    ep_data['multiplier']    = multiplier
    ep_data['dynamic_price'] = BASELINE_RATE * multiplier
    ep_data['tier']          = tier

    # Demand response via price elasticity
    ep_data['adjusted_demand'] = (
        ep_data['kWhDelivered'] * (1 + PRICE_ELASTICITY * (ep_data['multiplier'] - 1.0))
    ).clip(lower=0)

    # ----------------------------------------------------------------
    # METRIC 1: Revenue Gain % (ACN, USD)
    # ----------------------------------------------------------------
    baseline_revenue    = ep_data['kWhDelivered'].sum() * BASELINE_RATE
    posted_dynamic_rev  = (ep_data['kWhDelivered'] * ep_data['dynamic_price']).sum()
    net_dynamic_rev     = (ep_data['adjusted_demand'] * ep_data['dynamic_price']).sum()
    posted_revenue_gain = ((posted_dynamic_rev - baseline_revenue) / baseline_revenue) * 100
    net_revenue_gain    = ((net_dynamic_rev - baseline_revenue) / baseline_revenue) * 100

    # ----------------------------------------------------------------
    # METRIC 2: Customer Response Rate
    # Demand retained after elasticity. This is a clearer customer response
    # proxy than "sessions exposed to any price change", which is reported
    # separately as price_exposure_rate.
    # ----------------------------------------------------------------
    sessions_with_price_change = (np.abs(ep_data['multiplier'] - 1.00) > 1e-6).sum()
    price_exposure_rate        = (sessions_with_price_change / len(ep_data)) * 100
    customer_response_rate     = (
        ep_data['adjusted_demand'].sum() / ep_data['kWhDelivered'].sum() * 100
    )

    # ----------------------------------------------------------------
    # METRIC 3: Pricing Efficiency Score (USD/kWh)
    # ----------------------------------------------------------------
    total_adjusted_kwh = ep_data['adjusted_demand'].sum()
    pricing_efficiency = (
        net_dynamic_rev / total_adjusted_kwh if total_adjusted_kwh > 0 else BASELINE_RATE
    )

    # ----------------------------------------------------------------
    # METRIC 4: Average Waiting Time Reduction
    # UrbanEV proxy from Agent 2 baseline vs simulated congested-hour utilization.
    # This is intentionally independent of ACN revenue-threshold learning because
    # the waiting-time KPI is specified on UrbanEV.
    # ----------------------------------------------------------------
    baseline_wait, post_pricing_wait, wait_time_reduction = evaluate_urbanev_wait_proxy(
        surge_thr, discount_thr
    )

    return {
        'posted_revenue_gain_pct': posted_revenue_gain,
        'net_revenue_gain_pct':    net_revenue_gain,
        'customer_response_rate':  customer_response_rate,
        'pricing_efficiency':      pricing_efficiency,
        'wait_time_reduction':     wait_time_reduction,
        'baseline_wait_index':      baseline_wait,
        'post_pricing_wait_index':  post_pricing_wait,
        'price_exposure_rate':      price_exposure_rate,
        'surge_pct':               (ep_data['tier'] == 'surge').mean()    * 100,
        'discount_pct':            (ep_data['tier'] == 'discount').mean() * 100,
        'avg_multiplier':          ep_data['multiplier'].mean()
    }


for episode in range(N_EPISODES):

    metrics = evaluate_thresholds(surge_threshold, discount_threshold)

    episode_results.append({
        'episode':            episode + 1,
        'surge_threshold':    surge_threshold,
        'discount_threshold': discount_threshold,
        **metrics
    })

    print(f"  Episode {episode+1:2d}: "
          f"Posted Rev {metrics['posted_revenue_gain_pct']:+.2f}% | "
          f"Net Rev {metrics['net_revenue_gain_pct']:+.2f}% | "
          f"Demand Retained {metrics['customer_response_rate']:.1f}% | "
          f"Exposure {metrics['price_exposure_rate']:.1f}% | "
          f"Efficiency ${metrics['pricing_efficiency']:.4f}/kWh | "
          f"Wait Reduction {metrics['wait_time_reduction']:.2f}% | "
          f"Surge {metrics['surge_pct']:.1f}% | "
          f"Thresholds: surge={surge_threshold:.3f}, discount={discount_threshold:.3f}")

    # ----------------------------------------------------------------
    # FEEDBACK LOOP: bounded local search across 9 candidate policies.
    # Objective = net revenue gain + congestion reduction - customer disruption penalty.
    # The selected policy may reduce the discount window when discounts are not
    # producing enough net value.
    # ----------------------------------------------------------------
    if episode < N_EPISODES - 1:
        candidate_policies = []

        # Wider search steps allow clearer learning trajectory across episodes
        for surge_step in [-0.05, 0.0, 0.05]:
            for discount_step in [-0.02, 0.0, 0.02]:
                c_surge    = round(min(0.75, max(0.575, surge_threshold   + surge_step)), 3)
                c_discount = round(min(0.35, max(0.10, discount_threshold + discount_step)), 3)
                c_metrics  = evaluate_thresholds(c_surge, c_discount)
                score = (
                    c_metrics['net_revenue_gain_pct']
                    + 1.00 * c_metrics['wait_time_reduction']
                    + 0.03 * (c_metrics['customer_response_rate'] - 95.0)
                    - 0.01 * c_metrics['price_exposure_rate']
                )
                candidate_policies.append((score, c_surge, c_discount))

        _, surge_threshold, discount_threshold = max(candidate_policies, key=lambda x: x[0])


In [ ]:
episodes_df = pd.DataFrame(episode_results)
episodes_df.to_csv(os.path.join(OUTPUT_DIR, 'monitoring_metrics.csv'), index=False)
print(f"\nSaved monitoring metrics for {N_EPISODES} episodes")
print(episodes_df[['episode', 'surge_threshold', 'discount_threshold',
                   'net_revenue_gain_pct', 'wait_time_reduction',
                   'customer_response_rate', 'price_exposure_rate',
                   'pricing_efficiency']].to_string(index=False))


---
## 6. Learning Curve Visualization


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Monitoring & Learning Agent -- Performance Across Episodes', fontsize=14, fontweight='bold')

# 1. Revenue Gain %
axes[0, 0].plot(episodes_df['episode'], episodes_df['posted_revenue_gain_pct'],
                marker='o', color=COLORS['primary'], linewidth=2, label='Posted tariff revenue')
axes[0, 0].plot(episodes_df['episode'], episodes_df['net_revenue_gain_pct'],
                marker='s', color=COLORS['dark'], linewidth=2, label='After elasticity response')
axes[0, 0].fill_between(episodes_df['episode'], episodes_df['net_revenue_gain_pct'],
                         alpha=0.15, color=COLORS['primary'])
axes[0, 0].axhline(y=0, color='black', linestyle='--', alpha=0.4, label='Flat baseline')
axes[0, 0].set_xlabel('Episode')
axes[0, 0].set_ylabel('Revenue Gain %')
axes[0, 0].set_title('Revenue Gain % Over Episodes (ACN, USD)')
axes[0, 0].legend()

# 2. Average Waiting Time Reduction (UrbanEV proxy)
axes[0, 1].plot(episodes_df['episode'], episodes_df['wait_time_reduction'],
                marker='D', color=COLORS['off_peak'], linewidth=2)
axes[0, 1].fill_between(episodes_df['episode'], episodes_df['wait_time_reduction'],
                         alpha=0.15, color=COLORS['off_peak'])
axes[0, 1].set_xlabel('Episode')
axes[0, 1].set_ylabel('Wait Time Reduction (%)')
axes[0, 1].set_title('Avg Waiting Time Reduction -- Congested Hours\n(UrbanEV utilization proxy)')
axes[0, 1].annotate(
    f"Baseline wait index: {baseline_peak_congestion:.3f}\n(UrbanEV congested-hour utilization)",
    xy=(0.05, 0.85), xycoords='axes fraction', fontsize=9,
    bbox=dict(boxstyle='round,pad=0.3', facecolor='wheat', alpha=0.5)
)

# 3. Pricing Efficiency Score
axes[0, 2].plot(episodes_df['episode'], episodes_df['pricing_efficiency'],
                marker='^', color=COLORS['peak'], linewidth=2)
axes[0, 2].fill_between(episodes_df['episode'], episodes_df['pricing_efficiency'],
                         alpha=0.15, color=COLORS['peak'])
axes[0, 2].axhline(y=BASELINE_RATE, color='black', linestyle='--', alpha=0.4,
                    label=f'Baseline ${BASELINE_RATE}/kWh')
axes[0, 2].set_xlabel('Episode')
axes[0, 2].set_ylabel('USD/kWh')
axes[0, 2].set_title('Pricing Efficiency Score (Revenue/kWh, ACN)')
axes[0, 2].legend()

# 4. Customer Response and Price Exposure
axes[1, 0].plot(episodes_df['episode'], episodes_df['customer_response_rate'],
                marker='s', color=COLORS['secondary'], linewidth=2, label='Demand retained')
axes[1, 0].plot(episodes_df['episode'], episodes_df['price_exposure_rate'],
                marker='o', color=COLORS['dark'], linewidth=2, linestyle='--', label='Price exposure')
axes[1, 0].fill_between(episodes_df['episode'], episodes_df['customer_response_rate'],
                         alpha=0.15, color=COLORS['secondary'])
axes[1, 0].axhline(y=95, color='red', linestyle='--', alpha=0.4, label='95% retention reference')
axes[1, 0].set_xlabel('Episode')
axes[1, 0].set_ylabel('Rate (%)')
axes[1, 0].set_title('Customer Response vs Price Exposure\n(ACN + elasticity)')
axes[1, 0].legend()

# 5. Threshold Evolution
axes[1, 1].plot(episodes_df['episode'], episodes_df['surge_threshold'],
                marker='o', color=COLORS['peak'],     linewidth=2, label='Surge Threshold')
axes[1, 1].plot(episodes_df['episode'], episodes_df['discount_threshold'],
                marker='s', color=COLORS['off_peak'], linewidth=2, label='Discount Threshold')
axes[1, 1].set_xlabel('Episode')
axes[1, 1].set_ylabel('Threshold Value')
axes[1, 1].set_title('Threshold Evolution (Feedback Loop Learning)')
axes[1, 1].legend()

# 6. Pricing tier distribution
axes[1, 2].stackplot(
    episodes_df['episode'],
    episodes_df['surge_pct'],
    100 - episodes_df['surge_pct'] - episodes_df['discount_pct'],
    episodes_df['discount_pct'],
    labels=['Surge', 'Normal', 'Discount'],
    colors=[COLORS['peak'], COLORS['shoulder'], COLORS['off_peak']],
    alpha=0.7
)
axes[1, 2].set_xlabel('Episode')
axes[1, 2].set_ylabel('% of Sessions')
axes[1, 2].set_title('Pricing Tier Distribution Over Episodes')
axes[1, 2].legend(loc='upper right')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'monitoring_learning_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved monitoring_learning_curves.png")


---
## 7. Final Agent Performance Summary


In [ ]:
print("=" * 60)
print("MONITORING & LEARNING AGENT -- FINAL PERFORMANCE")
print("=" * 60)

final   = episodes_df.iloc[-1]
initial = episodes_df.iloc[0]

print(f"\n--- Performance: Episode 1 -> Episode {N_EPISODES} ---")
print(f"  Posted Revenue Gain %:   {initial['posted_revenue_gain_pct']:+.2f}%  ->  {final['posted_revenue_gain_pct']:+.2f}%")
print(f"  Net Revenue Gain %:      {initial['net_revenue_gain_pct']:+.2f}%  ->  {final['net_revenue_gain_pct']:+.2f}%")
print(f"  Demand Retained:         {initial['customer_response_rate']:.1f}%   ->  {final['customer_response_rate']:.1f}%")
print(f"  Price Exposure:          {initial['price_exposure_rate']:.1f}%   ->  {final['price_exposure_rate']:.1f}%")
print(f"  Pricing Efficiency:      ${initial['pricing_efficiency']:.4f}  ->  ${final['pricing_efficiency']:.4f}/kWh")
print(f"  Wait Time Reduction:     {initial['wait_time_reduction']:.2f}%  ->  {final['wait_time_reduction']:.2f}%")
print(f"  Surge Sessions:          {initial['surge_pct']:.1f}%   ->  {final['surge_pct']:.1f}%")

print(f"\n--- Threshold Evolution ---")
print(f"  Surge threshold:         {initial['surge_threshold']:.3f}  ->  {final['surge_threshold']:.3f}")
print(f"  Discount threshold:      {initial['discount_threshold']:.3f}  ->  {final['discount_threshold']:.3f}")

print(f"\n--- Final Pricing Tier Distribution ---")
print(f"  Surge sessions:    {final['surge_pct']:.1f}%")
print(f"  Normal sessions:   {100 - final['surge_pct'] - final['discount_pct']:.1f}%")
print(f"  Discount sessions: {final['discount_pct']:.1f}%")


### 7.1 Comprehensive Metrics Table (Submission-ready)


In [ ]:
# Read best model name from model_comparison.csv
model_comp_path = os.path.join(OUTPUT_DIR, 'model_comparison.csv')
if os.path.exists(model_comp_path):
    model_comp = pd.read_csv(model_comp_path, index_col=0)
    best_model_name = model_comp['R2'].astype(float).idxmax()
    best_rmse = float(model_comp.loc[best_model_name, 'RMSE'])
    best_mae  = float(model_comp.loc[best_model_name, 'MAE'])
    best_r2   = float(model_comp.loc[best_model_name, 'R2'])
else:
    best_model_name, best_rmse, best_mae, best_r2 = "XGBoost", "N/A", "N/A", "N/A"

# Read revenue comparison for accurate ACN revenue gain
rev_comp_path = os.path.join(OUTPUT_DIR, 'revenue_comparison.csv')
if os.path.exists(rev_comp_path):
    rev_comp = pd.read_csv(rev_comp_path)
    rev_gain_row = rev_comp[rev_comp['Metric'] == 'Revenue Gain %']
    acn_revenue_gain = rev_gain_row['Value'].values[0] if len(rev_gain_row) > 0 else "N/A"
else:
    acn_revenue_gain = "N/A"

final_metrics = pd.DataFrame({
    'Metric': [
        'Best Demand Prediction Model',
        f'RMSE ({best_model_name})',
        f'MAE ({best_model_name})',
        f'R2 Score ({best_model_name})',
        'Revenue Gain % -- Episode 1 (ACN, USD, posted)',
        f'Revenue Gain % -- Episode {N_EPISODES} (ACN, USD, posted)',
        f'Net Revenue Gain % -- Episode {N_EPISODES} (after elasticity)',
        'Revenue Gain % -- Flat comparison (Notebook 3)',
        f'Off-Peak Uplift % (UrbanEV, Notebook 3)',
        f'Avg Waiting Time Reduction % -- Episode 1',
        f'Avg Waiting Time Reduction % -- Final Episode',
        f'Customer Response Rate % -- Episode 1 (demand retained)',
        f'Customer Response Rate % -- Final Episode (demand retained)',
        f'Price Exposure Rate % -- Episode 1',
        f'Price Exposure Rate % -- Final Episode',
        f'Pricing Efficiency -- Baseline (USD/kWh)',
        f'Pricing Efficiency -- Final Episode (USD/kWh)',
        'Final Surge Threshold',
        'Final Discount Threshold',
        'Number of Learning Episodes',
        'Price Elasticity Assumption',
        'UrbanEV Baseline Peak Congestion (avg utilization)',
        'UrbanEV Post-Pricing Peak Congestion (avg utilization)',
    ],
    'Value': [
        best_model_name,
        f"{best_rmse:.4f}" if isinstance(best_rmse, float) else best_rmse,
        f"{best_mae:.4f}"  if isinstance(best_mae, float)  else best_mae,
        f"{best_r2:.4f}"   if isinstance(best_r2, float)   else best_r2,
        f"{initial['posted_revenue_gain_pct']:+.2f}%",
        f"{final['posted_revenue_gain_pct']:+.2f}%",
        f"{final['net_revenue_gain_pct']:+.2f}%",
        acn_revenue_gain,
        OFF_PEAK_UPLIFT_FROM_NB3,
        f"{initial['wait_time_reduction']:.2f}%",
        f"{final['wait_time_reduction']:.2f}%",
        f"{initial['customer_response_rate']:.1f}%",
        f"{final['customer_response_rate']:.1f}%",
        f"{initial['price_exposure_rate']:.1f}%",
        f"{final['price_exposure_rate']:.1f}%",
        f"${BASELINE_RATE:.4f}",
        f"${final['pricing_efficiency']:.4f}",
        f"{final['surge_threshold']:.3f}",
        f"{final['discount_threshold']:.3f}",
        str(N_EPISODES),
        "-0.30 (conservative, EV charging literature)",
        f"{final['baseline_wait_index']:.4f}",
        f"{final['post_pricing_wait_index']:.4f}",
    ],
    'Dataset_Note': [
        'UrbanEV (training)', 'UrbanEV', 'UrbanEV', 'UrbanEV',
        'ACN (USD)', 'ACN (USD)', 'ACN + elasticity',
        'ACN (USD)', 'UrbanEV (CNY)',
        'UrbanEV congestion proxy', 'UrbanEV congestion proxy',
        'ACN + elasticity', 'ACN + elasticity',
        'ACN dynamic tariff exposure', 'ACN dynamic tariff exposure',
        'ACN (USD)', 'ACN (USD)',
        'ACN (USD)', 'ACN (USD)',
        '-', '-', 'UrbanEV', 'UrbanEV'
    ]
})

print("\n=== FINAL METRICS TABLE ===")
print(final_metrics.to_string(index=False))
final_metrics.to_csv(os.path.join(OUTPUT_DIR, 'agent_performance_summary.csv'), index=False)
print("\nSaved agent_performance_summary.csv")


### 7.2 Agent 3 KPI Export


In [ ]:
agent3_kpi = pd.DataFrame({
    'Agent': ['Agent 3 (Monitoring)'] * 7,
    'KPI': [
        'Avg Waiting Time Reduction % (Final Episode)',
        'Customer Response Rate % (Final Episode, demand retained)',
        'Price Exposure Rate % (Final Episode)',
        'Pricing Efficiency Score - Final (USD/kWh)',
        'Net Revenue Gain % (Final Episode)',
        'UrbanEV Baseline Peak Congestion',
        'UrbanEV Post-Pricing Peak Congestion',
    ],
    'Value': [
        f"{final['wait_time_reduction']:.2f}%",
        f"{final['customer_response_rate']:.1f}%",
        f"{final['price_exposure_rate']:.1f}%",
        f"${final['pricing_efficiency']:.4f}",
        f"{final['net_revenue_gain_pct']:+.2f}%",
        f"{final['baseline_wait_index']:.4f} (avg utilization)",
        f"{final['post_pricing_wait_index']:.4f} (avg utilization)",
    ],
    'Dataset': [
        'UrbanEV congestion proxy',
        'ACN + demand elasticity',
        'ACN dynamic tariff exposure',
        'ACN (USD)',
        'ACN + demand elasticity',
        'UrbanEV',
        'UrbanEV',
    ]
})
agent3_kpi.to_csv(os.path.join(OUTPUT_DIR, 'agent3_kpi.csv'), index=False)
print("Saved agent3_kpi.csv")


---
## 8. Key Takeaways

1. **Feedback loop improves revenue and efficiency:** Bounded local search across
   9 candidate policies moves the system toward better net-revenue settings.
   In this ACN run, the biggest improvement comes from narrowing the discount
   window, not from creating many surge sessions.

2. **Wait time reduction is measured on UrbanEV directly:**
   The wait proxy compares baseline vs simulated utilization in the UrbanEV
   hours with the highest baseline congestion. This avoids mixing ACN demand
   deflection with the UrbanEV queue metric.

3. **Customer response remains high:** Customer response is reported as demand
   retained after elasticity, while price exposure is reported separately.
   This avoids presenting lower exposure as a worse customer outcome.

4. **All assumptions documented:**
   - Elasticity = -0.30 (conservative, consistent across all agents)
   - Congestion proxy = UrbanEV high-congestion-hour utilization (meaningful, 0-1 range)
   - ACN utilization estimated from hourly session counts / true max capacity
   - Currencies kept separate: USD (ACN), CNY (UrbanEV)
   - Baseline: $0.30/kWh USD equivalent of Rs.15/kWh Indian market rate

5. **Limitations acknowledged:**
   - True demand elasticity for Indian/Chinese markets may differ
   - Wait time proxy is indirect (no direct wait-time data in either dataset)
   - Episode simulation is sequential on same dataset (not true online learning)
   - ACN is a US workplace dataset -- naturally lower utilization limits surge events
